# Agents in LangChain
### Converted from OpenAI → Google Gemini
**ReAct Agent using `gemini-2.5-flash` + DuckDuckGo Search + Weather API**

---

## Step 1 — Install Dependencies

In [ ]:
!pip install -q langchain-google-genai langchain-community langchain-core langchain requests duckduckgo-search

## Step 2 — API Keys

In [ ]:
import os

os.environ['GEMINI_API_KEY'] = 'your_gemini_api_key_here'

## Step 3 — Imports

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
import requests
import os

## Step 4 — Define Tools

Same two tools as the original:
- `DuckDuckGoSearchRun` — searches the web
- `get_weather_data` — fetches live weather from Weatherstack API

In [ ]:
# Built-in search tool
search_tool = DuckDuckGoSearchRun()

print(f'Search tool ready: {search_tool.name}')

In [ ]:
# Custom weather tool
@tool
def get_weather_data(city: str) -> str:
    """
    This function fetches the current weather data for a given city
    """
    url = f'https://api.weatherstack.com/current?access_key=4d1d8ae207a8c845a52df8a67bf3623e&query={city}'
    response = requests.get(url)
    return str(response.json())

print(f'Weather tool ready: {get_weather_data.name}')

## Step 5 — Initialise Gemini Model

Replacing `ChatOpenAI()` with `ChatGoogleGenerativeAI` using `gemini-2.5-flash`.

In [ ]:
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=os.environ['GEMINI_API_KEY']
)

print('Model ready: gemini-2.5-flash')

## Step 6 — Pull the ReAct Prompt from LangChain Hub

`hwchase17/react` is the standard ReAct prompt used for all ReAct agents.
This is the same as the original — no change needed here.

In [ ]:
prompt = hub.pull('hwchase17/react')

print('ReAct prompt pulled successfully')
print(prompt.input_variables)

## Step 7 — Create the ReAct Agent

Same `create_react_agent()` call as original, just using Gemini instead of OpenAI.

In [ ]:
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

print('ReAct agent created')

## Step 8 — Wrap with AgentExecutor

`AgentExecutor` runs the agent loop automatically.
`verbose=True` prints every thought, action, and observation — exactly like the original output you saw.

In [ ]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True,
    handle_parsing_errors=True
)

print('AgentExecutor ready')

## Step 9 — Run the Agent!

Same query as original — find the capital of Madhya Pradesh, then get its weather.

In [ ]:
response = agent_executor.invoke({
    'input': 'Find the capital of Madhya Pradesh, then find its current weather condition'
})

print(response['output'])

## Step 10 — Try More Queries

The agent can chain any number of tool calls. Try these:

In [ ]:
# Try 2 — different city
response2 = agent_executor.invoke({
    'input': 'What is the capital of Tamil Nadu and what is the weather there right now?'
})
print(response2['output'])

In [ ]:
# Try 3 — search + weather combined
response3 = agent_executor.invoke({
    'input': 'Which city hosted IPL 2024 finals and what is the current weather in that city?'
})
print(response3['output'])

---
## What Changed from the Original

| Original (OpenAI) | Converted (Gemini) |
|---|---|
| `from langchain_openai import ChatOpenAI` | `from langchain_google_genai import ChatGoogleGenerativeAI` |
| `os.environ['OPENAI_API_KEY'] = ...` | `os.environ['GEMINI_API_KEY'] = ...` |
| `llm = ChatOpenAI()` | `llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', google_api_key=...)` |
| `!pip install langchain-openai` | `!pip install langchain-google-genai` |

Everything else — `create_react_agent`, `AgentExecutor`, `hub.pull`, `@tool`, `DuckDuckGoSearchRun` — stays **exactly the same**.

---
## How the ReAct Agent Works

ReAct stands for **Reason + Act**. The agent follows this loop internally:

```
Thought: I need to find the capital of Madhya Pradesh first
Action: duckduckgo_search
Action Input: capital of Madhya Pradesh
Observation: Bhopal is the capital...
Thought: Now I need the weather for Bhopal
Action: get_weather_data
Action Input: Bhopal
Observation: {temperature: 40, ...}
Thought: I have everything I need
Final Answer: The capital is Bhopal, currently 40°C and partly cloudy.
```

The `verbose=True` in `AgentExecutor` prints this entire reasoning process so you can see exactly how the agent thinks step by step. This is the same loop you built manually in the tool calling lecture — `AgentExecutor` just automates it.